# Phoenix, Arizona — Land Value Tax (4:1 Split-Rate) Model

Models a revenue-neutral **City of Phoenix primary property tax** shift to a 4:1
split-rate (land taxed at 4× the improvement rate), preserving Arizona's existing
legal-class assessment ratios, the Limited Property Value (Prop 117) base, and all
current exemptions.

**Scope decisions (confirmed up front):**
- **Levy modeled:** City of Phoenix *primary* levy only (rate $1.2658 per $100 of
  assessed valuation = 12.658 mills). County, school, community-college, and
  secondary/bond levies are excluded.
- **Reform:** 4:1 split-rate, revenue-neutral against the modeled current primary levy.
- **Exemptions / classes:** preserved. Tax base is the assessor's pre-computed
  **Limited Property Value Assessed** (`LPV_Assessed` = LPV × statutory class ratio),
  exactly as Arizona levies the primary tax. Government / religious / charitable
  parcels (PUC 90–99) are excluded as exempt.

**Data sources (both from the Maricopa County Assessor, tax year 2027):**
1. **Parcel geometries** — Assessor Parcels MapServer, filtered server-side to
   `JURISDICTION='PHOENIX'` (`data/phoenix_parcels.gpq`, ~525k parcels).
2. **Tax roll** — Assessor "Secured Master" bulk download (pipe-delimited, `data/Data/`),
   joined to geometries on APN. Provides `Land_Full_Cash_Value`,
   `Improvement_Full_Cash_Value`, `LPV_Assessed`, `Property_Use_Code`, legal class.

**Land/improvement split:** `LPV_Assessed` is allocated to land vs. improvement by each
parcel's full-cash-value ratio (`Land_FCV / (Land_FCV + Improvement_FCV)`). This preserves
the class-ratio weighting baked into `LPV_Assessed` (the same approach used for St. Paul's
Minnesota Tax Capacity), then applies the split-rate to those assessed components.

In [ ]:
import sys
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# Constants
CITY_NAME = 'phoenix'
STATE_FIPS = '04'
COUNTY_FIPS = '013'
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

# City of Phoenix primary property tax rate, $1.2658 per $100 of assessed value.
# calculate_current_tax expects mills (per $1,000): $1.2658/$100 = 12.658 / $1,000.
PRIMARY_RATE_PER_100 = 1.2658
PRIMARY_MILLAGE = PRIMARY_RATE_PER_100 * 10  # = 12.658 mills

# Official validation target — Maricopa County 2025 tax levy (FY2025-26):
#   Phoenix primary net assessed valuation = $17,772,778,262
#   Phoenix primary levy = $17,772,778,262 x 1.2658/100 = $224,963,627
# Our parcel data is tax year 2027, so modeled revenue should land slightly above
# this (modest Prop-117-capped LPV growth) — a small positive gap is expected.
OFFICIAL_PRIMARY_NAV = 17_772_778_262
OFFICIAL_PRIMARY_LEVY = OFFICIAL_PRIMARY_NAV * PRIMARY_RATE_PER_100 / 100

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 1 — Load parcel geometries + Secured Master tax roll

Geometries were pre-fetched from the Maricopa Assessor Parcels MapServer
(`WHERE JURISDICTION='PHOENIX'`) by `data/fetch_phoenix_geometry.py`. The tax roll is the
Assessor's "Secured Master" bulk download (five Book-series files), joined on APN.

In [ ]:
PARCEL_PATH = DATA_DIR / 'phoenix_parcels.gpq'
gdf = gpd.read_parquet(PARCEL_PATH)
# Keep only APN + geometry; the Secured Master tax roll (below) is the authoritative
# source for all valuation/use/class fields, so we avoid duplicate-column collisions.
gdf = gdf[['APN', 'geometry']].copy()
print(f"Loaded {len(gdf):,} Phoenix-jurisdiction parcels (geometry) from cache")

# Secured Master: pipe-delimited, no header, latin-1, 39 columns (see data/File Spec/)
SM_COLS = [
    "Parcel_Number", "Owner_Name", "Owner_Mail_Address_1", "Owner_Mail_Address_2",
    "Owner_Mail_City", "Owner_Mail_State", "Owner_Mail_Zip", "Owner_Mail_Country",
    "Situs_Address", "Situs_Suite", "Situs_City", "Situs_Zip",
    "Property_Type", "Deed_Number", "Deed_Date", "Deed_Type",
    "Land_FCV", "Improvement_FCV", "Total_FCV", "Total_FCV_Assessed",
    "LPV", "LPV_Assessed",
    "PUC", "Lot", "Block", "Tract",
    "Land_Class", "Improvement_Class", "Rental_Indicator",
    "Living_Area", "Total_SqFt", "Construction_Year", "Pool_Size",
    "Sale_Price", "Sale_Date", "MCR_Number", "Subdivision_Name",
    "Number_Of_Units", "Tax_Area_Code",
]
NUM_COLS = ["Land_FCV", "Improvement_FCV", "Total_FCV", "Total_FCV_Assessed", "LPV", "LPV_Assessed"]
KEEP = ["Parcel_Number", "Property_Type", "PUC", "Land_Class", "Tax_Area_Code",
        "Construction_Year", "Number_Of_Units"] + NUM_COLS

_phoenix_apns = set(gdf['APN'])
_frames = []
for _bk in [100, 200, 300, 400, 500]:
    _df = pd.read_csv(
        DATA_DIR / 'Data' / f'Secured_Master_BK{_bk}.txt',
        sep='|', header=None, names=SM_COLS, dtype=str,
        encoding='latin-1', low_memory=False,
    )
    _df = _df[_df['Parcel_Number'].isin(_phoenix_apns)]
    _frames.append(_df[KEEP])
roll = pd.concat(_frames, ignore_index=True)
for _c in NUM_COLS:
    roll[_c] = pd.to_numeric(roll[_c], errors='coerce').fillna(0.0)

# Join tax roll onto geometries by APN
gdf = gdf.merge(roll, left_on='APN', right_on='Parcel_Number', how='inner')
print(f"Joined tax roll: {len(gdf):,} parcels ({gdf['Parcel_Number'].notna().mean()*100:.2f}% matched)")
print(f"Total LPV_Assessed: ${gdf['LPV_Assessed'].sum():,.0f}")

### Column mapping

| Concept | Column | Notes |
|---|---|---|
| Land value (market) | `Land_FCV` | Full Cash Value of land |
| Improvement value (market) | `Improvement_FCV` | Full Cash Value of improvements |
| Tax base (assessed) | `LPV_Assessed` | Limited Property Value × statutory class ratio — the value Arizona's primary tax is levied on (Prop 117) |
| Use code | `PUC` | 4-digit Arizona Property Use Code |
| Legal class | `Land_Class` | e.g. `100% 3.1` (owner-occupied residential, 10% ratio) |
| Exemption | PUC prefix `90`–`99` | Government / religious / charitable → fully exempt |
| Parcel ID | `APN` | Join key between geometry and tax roll |

**Assessment ratios** are already embedded in `LPV_Assessed` (verified: Class 3 = 10%,
Class 1 ≈ 15%, Class 2 = 15%, Class 6 = 5%). **Millage source:** City of Phoenix adopted
primary rate, $1.2658/$100 (Maricopa County 2025 Tax Levy schedule).

## Section 2 — Classify property types and flag exemptions

Categories are derived from the Arizona Property Use Code (PUC). The first two digits give
the major use; code `03` (multiple residential) is split by its third digit into 2–4 unit
vs. 5+ unit. Government / religious / charitable property (PUC 90–99) is flagged exempt and
excluded from the revenue-neutral base.

In [ ]:
def phoenix_category(puc: str) -> str:
    """Map a 4-digit Arizona Property Use Code to a standard LVTShift category."""
    p = (puc or '').strip()
    p2 = p[:2]
    p3 = p[:3]
    # Vacant
    if p2 == '00':
        return 'Vacant Land'
    # Residential
    if p2 in ('01', '87'):
        return 'Single Family Residential'
    if p2 == '07':
        return 'Condominium'
    if p2 == '03':
        # 031 mixed / 032 duplex / 033 triplex / 034 fourplex -> 2-4 units; 035+ -> 5+ units
        return 'Small Multi-Family (2-4 units)' if p3 in ('031', '032', '033', '034') \
            else 'Large Multi-Family (5+ units)'
    if p2 in ('02', '08', '85'):
        return 'Other Residential'  # PUD common areas, manufactured-home land, mobile homes
    # Lodging
    if p2 in ('04', '05', '06'):
        return 'Hotel'
    # Commercial
    if p2 in ('11', '12', '13', '14', '15'):
        return 'Retail / General Commercial'
    if p2 == '16':
        return 'Office / Commercial Condo'
    if p2 == '27':
        return 'Transportation - Parking'
    if p2 in ('10', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '28', '30'):
        return 'Other Commercial'
    if p2 in ('37', '38'):
        return 'Industrial'
    # Agricultural
    if p2 in ('40', '41', '42', '43', '44', '45', '46', '47', '48', '49'):
        return 'Agricultural'
    # Centrally valued (utilities, railroads, mines, pipelines)
    if p2 in tuple(f'{i:02d}' for i in range(50, 70)):
        return 'Other Commercial'
    # Exempt government / religious / charitable
    if p2 in tuple(f'{i:02d}' for i in range(90, 100)):
        return 'Exempt'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf['PUC'].map(phoenix_category)

# Exemption flag: PUC 90-99 (govt / religious / charitable) -> not part of the taxable base
gdf['full_exmp'] = gdf['PUC'].fillna('').str[:2].isin(
    [f'{i:02d}' for i in range(90, 100)]
).astype(int)

# Override: any taxable parcel with $0 improvement value is Vacant Land
gdf.loc[(gdf['full_exmp'] == 0) & (gdf['Improvement_FCV'] <= 0), 'PROPERTY_CATEGORY'] = 'Vacant Land'

print(f"Exempt parcels (PUC 90-99): {gdf['full_exmp'].sum():,}")
print(f"Taxable parcels:            {(gdf['full_exmp'] == 0).sum():,}")
print()
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

## Section 3 — Current primary tax & revenue validation

The City of Phoenix primary tax is levied on `LPV_Assessed` at $1.2658 / $100 (12.658
mills). Exempt parcels (PUC 90–99) contribute zero. Modeled revenue is validated against
the Maricopa County 2025 published levy.

In [ ]:
gdf['millage_rate'] = PRIMARY_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='LPV_Assessed',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_PRIMARY_LEVY - 1) * 100
print(f"Modeled primary revenue: ${current_revenue:,.0f}")
print(f"Official 2025 levy:      ${OFFICIAL_PRIMARY_LEVY:,.0f}  "
      f"(NAV ${OFFICIAL_PRIMARY_NAV:,.0f} x {PRIMARY_RATE_PER_100}/100)")
print(f"Gap:                     {gap_pct:+.2f}%   "
      f"(positive gap expected: data is TY2027 vs. official TY2025)")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds 5% threshold"

## Section 4 — Revenue-neutral 4:1 split-rate model

`LPV_Assessed` is allocated to land vs. improvement by the parcel's full-cash-value land
ratio. Vacant parcels (no improvement value) put their entire base on land. The split-rate
solver then finds revenue-neutral land and improvement millages with land taxed at 4× the
improvement rate, across taxable parcels only.

In [ ]:
# Allocate the assessed tax base to land vs improvement by FCV composition
_fcv_total = gdf['Land_FCV'] + gdf['Improvement_FCV']
gdf['land_value_ratio'] = np.where(_fcv_total > 0, gdf['Land_FCV'] / _fcv_total, 1.0)
gdf['taxable_land_value'] = gdf['land_value_ratio'] * gdf['LPV_Assessed']
gdf['taxable_improvement_value'] = (1 - gdf['land_value_ratio']) * gdf['LPV_Assessed']

# Exempt parcels carry no taxable base
gdf.loc[gdf['full_exmp'] == 1, ['taxable_land_value', 'taxable_improvement_value']] = 0.0

# Model split-rate on taxable parcels only
taxable = gdf[gdf['full_exmp'] == 0].copy()
land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels (new_tax = current_tax = 0)
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()
gdf['tax_change'] = gdf['new_tax'] - gdf['current_tax']
gdf['tax_change_pct'] = np.where(
    gdf['current_tax'] > 0, gdf['tax_change'] / gdf['current_tax'] * 100, 0.0
)

print(f"Land millage:        {land_millage:.4f}")
print(f"Improvement millage: {improvement_millage:.4f}  (ratio {land_millage/improvement_millage:.2f}:1)")
print(f"New revenue:  ${new_revenue:,.0f}")
print(f"Current revenue: ${taxable['current_tax'].sum():,.0f}  (revenue-neutral check)")

In [ ]:
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact")

## Section 5 — Exploration: median tax change by category

In [ ]:
matplotlib.use('Agg')  # headless
fig, ax = plt.subplots(figsize=(10, 6))
_summary = (
    gdf[gdf['full_exmp'] == 0]
    .groupby('PROPERTY_CATEGORY')['tax_change_pct']
    .median()
    .sort_values()
)
_summary.plot.barh(ax=ax, color='#2a7fb8')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'{CITY_NAME.title()} — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('Saved category_preview.png')

## Section 6 — Census join + standardized export + report

Maricopa County (FIPS 04013) block-group demographics were pre-fetched and cached by
`data/fetch_census.py` (the county is large enough that the live TIGER fetch is chunked
tract-by-tract). The cell loads the cache when present, otherwise falls back to a live
fetch with a generous timeout.

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_bg_cache = DATA_DIR / 'census_bg.gpq'
_fips = STATE_FIPS + COUNTY_FIPS
try:
    if _bg_cache.exists():
        census_gdf = gpd.read_parquet(_bg_cache)
        print(f"Loaded {len(census_gdf):,} census block groups from cache")
    else:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            _census_data, census_gdf = _future.result(timeout=600)
    gdf = match_to_census_blockgroups(gdf, census_gdf)
    # census_gdf already carries demographics from get_census_data_with_boundaries — the
    # spatial join brings them onto gdf. Do NOT merge census_data again (creates _x/_y dupes).
    if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
        gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
    if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
        gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
    print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
except concurrent.futures.TimeoutError:
    print("Census API timed out — skipping census join")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    exempt_flag_col='full_exmp',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — PNGs in analysis/reports/phoenix/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

In [ ]:
# --- Parcel-map export (interactive tax-change map) ---
# Uses the SAME frame (gdf) and the SAME model_type / millages / tax columns as
# the save_standard_export call above, so the map matches the CSV exactly.
import geopandas as _gpd
from lvt.parcel_map import save_parcel_map_export, create_parcel_map

# gdf still carries WGS84 geometry; guard in case concat dropped the GeoDataFrame type.
_map_gdf = gdf if isinstance(gdf, _gpd.GeoDataFrame) else _gpd.GeoDataFrame(
    gdf, geometry='geometry', crs='EPSG:4326')

_map_out = save_parcel_map_export(
    gdf=_map_gdf,
    city='phoenix',
    output_path='../../analysis/maps/phoenix.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col='APN',
    parcel_url_template=None,
    owner_name_col=None,
    owner_address_col=None,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    exempt_flag_col='full_exmp',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Very large city (~525k parcels) — simplify the viewer geometry at 8 m.
create_parcel_map(_map_out, 'phoenix', simplify_tolerance_m=8)
print('phoenix parcel map done.')

## Validation Summary

| Check | Result |
|---|---|
| Revenue match | +2.22% — modeled $229.97M vs. official $224.97M (Maricopa County 2025 Tax Levy: Phoenix primary NAV $17.77B × $1.2658/$100). Positive gap expected: parcel data is tax year 2027, official levy is tax year 2025; Prop-117-capped LPV growth accounts for the difference. |
| Parcel count | 525,042 Phoenix-jurisdiction parcels (geometry joined to tax roll at 100%). 514,492 taxable, 10,550 exempt (PUC 90–99). |
| Census coverage | 100% spatial join to Maricopa block groups; 95.6% have ACS income, 99.4% minority %. |
| PNGs generated | 7 of 7 |
| Vacant Land median change | +126.5% |
| Single Family Residential median change | -9.4% |
| Commercial (retail/office) median change | +30% to +39% |

**Distributional read:** the 4:1 split-rate shifts burden off improvements and onto land, as
designed. Vacant and underused land (parking, vacant lots) sees the largest increases;
improvement-heavy single-family, condo, and multifamily residential see a modest ~9% cut.
Commercial and office rise because of their higher land share. Revenue is held exactly neutral.